### **OpenCLIP con Docker, una GPU y plantillas SLURM**

Este cuaderno es el **centro didáctico** del proyecto, no toda la lógica del proyecto.

Su función es:

- explicar la lógica experimental,
- inspeccionar el subconjunto real ya incluido,
- correr el baseline preentrenado,
- revisar retrieval y hard negatives,
- mostrar cómo se escala a `torchrun` y SLURM,
- y dejar el pipeline automatizado en `scripts/` y `src/`.

#### **Idea principal**
El flujo realista de trabajo es:

1. trabajar primero con un checkpoint preentrenado,
2. extraer embeddings de un subconjunto real,
3. evaluar recuperación cruzada,
4. inspeccionar errores,
5. dejar el fine-tuning CSV como extensión controlada.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Ejecuta este cuaderno desde la carpeta Semana4/")

sys.path.append(str(PROJECT_ROOT))

from src.io_utils import load_yaml
from src.dataset_utils import load_metadata
from src.openclip_utils import create_model, encode_image_paths, encode_texts
from src.metrics import summarize_ranking
from src.retrieval import topk_text_to_image, topk_image_to_text, mine_hard_negatives
from src.visualize import show_gallery, show_retrieval_results


#### **1. Cargar configuración local**


In [ ]:
cfg = load_yaml(PROJECT_ROOT / "configs/local.yaml")
cfg


#### **2. Verificación rápida del entorno**

Este cuaderno está alineado con el flujo Docker del curso.  
Antes de correr experimentos, conviene verificar:

- versión de PyTorch,
- visibilidad de CUDA,
- nombre de GPU,
- disponibilidad de `open_clip`.


In [ ]:
!python scripts/00_verify_env.py


#### **3. Inspección del subconjunto real ya incluido**

Aquí usamos un **bootstrap sample real de Flickr30k** ya materializado en el proyecto.

No es un dataset inventado: las imágenes provienen del dataset Flickr30k y sirven para:
- ver ejemplos reales,
- probar retrieval,
- estudiar hard negatives,
- demostrar la tubería sin depender de una descarga grande inicial.


In [ ]:
metadata = load_metadata(PROJECT_ROOT / "data/bootstrap_flickr30k/metadata.csv", root=PROJECT_ROOT)
metadata[["image_id", "label", "caption"]]


In [ ]:
fig = show_gallery(metadata, ncols=3, figsize=(12, 8))
plt.show()


#### **4. Cargar OpenCLIP**

Usaremos un baseline sólido y razonable para una sola GPU:

- `ViT-B-32`
- `laion2b_s34b_b79k`

Es un punto de partida realista para Semana 4: contraste, alineamiento y retrieval.


In [ ]:
model_name = cfg["model"]["model_name"]
pretrained = cfg["model"]["pretrained"]

model, preprocess, tokenizer, device = create_model(model_name, pretrained)
print("device =", device)
print("model_name =", model_name)
print("pretrained =", pretrained)


#### **5. Extraer embeddings del subconjunto bootstrap**


In [ ]:
image_features = encode_image_paths(
    model,
    preprocess,
    metadata["filepath"].tolist(),
    device=device,
    batch_size=cfg["runtime"]["batch_size"],
)

text_features = encode_texts(
    model,
    tokenizer,
    metadata["caption"].tolist(),
    device=device,
    batch_size=max(cfg["runtime"]["batch_size"], 32),
)

image_features.shape, text_features.shape


#### **6. Matriz de similitud y métricas de retrieval**

Como el bootstrap tiene pares alineados `(imagen_i, caption_i)`, la diagonal representa el match correcto.


In [ ]:
sim = image_features @ text_features.T
metrics_i2t = summarize_ranking(sim)
metrics_t2i = summarize_ranking(sim.T)

pd.DataFrame([
    {"direction": "image_to_text", **{k: v for k, v in metrics_i2t.items() if k != "Ranks"}},
    {"direction": "text_to_image", **{k: v for k, v in metrics_t2i.items() if k != "Ranks"}},
])


#### **7. Ejemplos de recuperación cruzada**


In [ ]:
query = "two men cooking in a kitchen"
query_feature = encode_texts(model, tokenizer, [query], device=device)
results = topk_text_to_image(query_feature, image_features, metadata, k=4)
results


In [ ]:
fig = show_retrieval_results(results, figsize=(12, 4))
plt.show()


In [ ]:
img_idx = 3
image_result = topk_image_to_text(image_features[img_idx:img_idx+1], text_features, metadata, k=4)
image_result


#### **8. Negativos duros**

Esta parte importa tanto como la métrica agregada. Los negativos duros muestran pares incorrectos con score alto y permiten discutir:

- correlaciones espurias,
- ambigüedad semántica,
- similitud superficial,
- límites del embedding compartido.


In [ ]:
hard = mine_hard_negatives(sim, metadata, top_n=8)
hard[["image_id", "image_label", "text_label", "score", "negative_caption"]]


In [ ]:
row = hard.iloc[0]
img = Image.open(row["image_filepath"]).convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f'Imagen: {row["image_label"]}\nNegative caption score={row["score"]:.3f}')
plt.axis("off")
plt.show()

print("Caption correcto:")
print(row["image_caption"])
print()
print("Hard negative:")
print(row["negative_caption"])


#### **9. Zero-shot como extensión**

El proyecto trae una extensión zero-shot mínima.
En este bootstrap la columna `label` es pequeña y curada solo para demostración.


In [ ]:
!python scripts/02_build_embeddings.py   --metadata-csv data/bootstrap_flickr30k/metadata.csv   --model-name ViT-B-32   --pretrained laion2b_s34b_b79k   --batch-size 16   --output outputs/embeddings/bootstrap_embeddings.npz


In [ ]:
!python scripts/04_eval_zeroshot.py   --embeddings outputs/embeddings/bootstrap_embeddings.npz   --metadata-csv data/bootstrap_flickr30k/metadata.csv   --prompt-config data/bootstrap_flickr30k/prompt_config.json   --output-csv outputs/metrics/zeroshot_predictions.csv


In [ ]:
pd.read_csv(PROJECT_ROOT / "outputs/metrics/zeroshot_predictions.csv")


#### **10. Pipeline reproducible fuera del notebook**

La idea correcta no es dejar todo dentro del notebook.

El proyecto ya trae pipeline reproducible:

- `scripts/run_local_pipeline.sh`
- `scripts/10_train_openclip_csv_local.sh`
- `scripts/11_train_openclip_csv_torchrun.sh`
- `slurm/train_openclip_csv_single_node.sbatch`
- `slurm/train_openclip_csv_multi_node.sbatch`

**Pipeline local completo**
```bash
bash scripts/run_local_pipeline.sh
```

**Fine-tuning CSV local muy corto**
```bash
bash scripts/10_train_openclip_csv_local.sh
```

**Template torchrun**
```bash
bash scripts/11_train_openclip_csv_torchrun.sh
```


#### **11. Descargar un subconjunto mayor y más realista de Flickr30k**

Cuando quieras salir del bootstrap y pasar a algo más serio, ejecuta:


In [ ]:
# Descomenta cuando quieras materializar un subconjunto mayor
# !python scripts/01_prepare_flickr30k_from_hf.py #   --output-root data/processed/flickr1k_hf #   --train-limit 512 #   --val-limit 50 #   --test-limit 50


In [ ]:
!python scripts/01_prepare_flickr30k_from_hf.py \
  --output-root data/processed/flickr1k_hf \
  --train-limit 512 \
  --val-limit 50 \
  --test-limit 50

Después puedes cambiar el CSV de entrada en `scripts/02_build_embeddings.py` y repetir la evaluación.

Eso te deja una progresión clara:

- bootstrap real incluido,
- subconjunto mayor descargado desde HF,
- evaluación reproducible,
- extensión a entrenamiento CSV,
- y plantillas listas para SLURM.


#### **13. Entregables sugeridos**

El estudiante debe entregar un notebook exportado. No se debe crear un archivo Markdown adicional dentro del proyecto.

El informe debe incluir:

1. Comandos ejecutados dentro del contenedor Docker.
2. Evidencia de ejecución del pipeline local con el bootstrap.
3. Evidencia de ejecución del pipeline remoto con `Vishva007/Flickr-Dataset-1k`, si hubo conexión a internet.
4. Tabla comparativa con `R@1`, `R@5`, `R@10` y `MRR`.
5. Análisis comentado de cinco hard negatives.
6. Comparación entre `--caption-mode first` y `--caption-mode all`.
7. Una mini ablación experimental.
8. Conclusiones técnicas y limitaciones del modelo.

##### **13.1 Experimentos mínimos**

Ejecutar el pipeline local dentro del contenedor:

```bash
cd /workspace/Semana4/Proyecto
bash scripts/run_local_pipeline.sh
```

Ejecutar el pipeline remoto dentro del contenedor:

```bash
cd /workspace/Semana4/Proyecto
bash scripts/run_hf_flickr1k_pipeline.sh
```

Comparar los resultados generados en:

```text
outputs/metrics/retrieval_metrics.json
outputs/metrics/hard_negatives.csv
outputs/metrics/flickr1k_retrieval_metrics.json
outputs/metrics/flickr1k_hard_negatives.csv
```

##### **13.2 Mini ablación sugerida**

El estudiante debe realizar al menos una de las siguientes variaciones.

**Comparar modos de captions**

Ejecutar una evaluación con una sola caption por imagen:

```bash
python scripts/02_build_embeddings.py \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --model-name ViT-B-32 \
  --pretrained laion2b_s34b_b79k \
  --batch-size 32 \
  --caption-mode first \
  --output outputs/embeddings/flickr1k_embeddings_first.npz

python scripts/03_eval_retrieval.py \
  --embeddings outputs/embeddings/flickr1k_embeddings_first.npz \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --output-json outputs/metrics/flickr1k_retrieval_metrics_first.json \
  --hard-negatives-csv outputs/metrics/flickr1k_hard_negatives_first.csv \
  --top-n-hard-negatives 20
```

Ejecutar una evaluación con todas las captions por imagen:

```bash
python scripts/02_build_embeddings.py \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --model-name ViT-B-32 \
  --pretrained laion2b_s34b_b79k \
  --batch-size 32 \
  --caption-mode all \
  --output outputs/embeddings/flickr1k_embeddings_all.npz

python scripts/03_eval_retrieval.py \
  --embeddings outputs/embeddings/flickr1k_embeddings_all.npz \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --output-json outputs/metrics/flickr1k_retrieval_metrics_all.json \
  --hard-negatives-csv outputs/metrics/flickr1k_hard_negatives_all.csv \
  --top-n-hard-negatives 20
```

**Cambiar el tamaño de batch**

Comparar al menos dos valores:

```bash
--batch-size 16
```

```bash
--batch-size 32
```

Reportar si cambia el tiempo de ejecución, el uso de memoria o la estabilidad del proceso.

**Comparar dos checkpoints de OpenCLIP**

Ejemplo base:

```bash
--model-name ViT-B-32
--pretrained laion2b_s34b_b79k
```

Ejemplo alternativo:

```bash
--model-name RN50
--pretrained openai
```

Reportar cuál obtiene mejores métricas y si los errores cualitativos cambian.

##### **13.3 Preguntas para el informe**

Responder brevemente:

1. ¿Qué cambia entre evaluar con una sola caption y evaluar con todas las captions asociadas a la misma imagen?,
2. ¿Qué tipo de errores aparecen en los hard negatives?,
3. ¿El modelo confunde objetos, acciones, contexto o relaciones espaciales?,
4. ¿Qué checkpoint obtiene mejores métricas?,
5. ¿Qué limitaciones tiene el bootstrap local frente a Flickr1k?,
6. ¿Qué mejora concreta aplicarías en una siguiente versión del laboratorio?.

##### **13.4 Formato recomendado de entrega**

Toda la documentación del proyecto debe mantenerse en `README.md`.

